## SECTION 1: Load Data & Exploratory Analysis

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
from pathlib import Path
from sklearn.base import clone

from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression, LinearRegression
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor, HistGradientBoostingRegressor
from sklearn.metrics import roc_auc_score, mean_absolute_error, mean_squared_error
from sklearn.inspection import permutation_importance

try:
    import torch
    import torch.nn as nn
    import torch.optim as optim
    HAS_TORCH = True
except Exception as e:
    HAS_TORCH = False
    print("PyTorch not available; MLP dropout learner will be skipped.", e)
    
try:
    import xgboost as xgb
    HAS_XGB = True
except:
    HAS_XGB = False
    
try:
    import lightgbm as lgb
    HAS_LGB = True
except:
    HAS_LGB = False
    
try:
    import catboost as cb
    HAS_CB = True
except:
    HAS_CB = False

from scipy import stats
from scipy.special import expm1

from collections import defaultdict
import itertools

warnings.filterwarnings('ignore')
np.random.seed(42)

print(f"XGBoost available: {HAS_XGB}")
print(f"LightGBM available: {HAS_LGB}")
print(f"CatBoost available: {HAS_CB}")

XGBoost available: True
LightGBM available: True
CatBoost available: True


In [2]:
data_path = '/home/kle-home-lab/code/HA-workspace/SC_Competition/DataSet_A.xlsx'
dict_path = '/home/kle-home-lab/code/HA-workspace/SC_Competition/data_dictionary_Challenge1.csv'
template_path = '/home/kle-home-lab/code/HA-workspace/SC_Competition/Treatment Effects.csv'

df = pd.read_excel(data_path)
data_dict = pd.read_csv(dict_path)
template = pd.read_csv(template_path)
df.columns = df.columns.str.upper()  

print(f"Dataset shape: {df.shape}")
print(f"Data dictionary shape: {data_dict.shape}")
print(f"Output template shape: {template.shape}")
print(f"\nFirst few columns: {df.columns[:10].tolist()}")

Dataset shape: (100000, 70)
Data dictionary shape: (70, 9)
Output template shape: (100000, 6)

First few columns: ['PAT_ID', 'Y1', 'Y2', 'Y3', 'Y4', 'Y5', 'T', 'X1', 'X2', 'X3']


In [3]:
assert df['PAT_ID'].nunique() == len(df), "PAT_ID is not unique!"

treatment_col = 'T'
print(f"\nTreatment distribution:")
print(df[treatment_col].value_counts())
print(f"Treatment ratio (T=1): {df[treatment_col].mean():.3f}")


Treatment distribution:
T
0    76402
1    23598
Name: count, dtype: int64
Treatment ratio (T=1): 0.236


In [4]:
outcome_cols = ['Y1', 'Y2', 'Y3', 'Y4', 'Y5']

print("Outcome ranges:")
for col in outcome_cols:
    if col in df.columns:
        print(f"{col}: [{df[col].min():.2f}, {df[col].max():.2f}], mean={df[col].mean():.2f}, missing={df[col].isna().sum()}")
    else:
        print(f"{col}: NOT FOUND")

Outcome ranges:
Y1: [0.30, 1.00], mean=0.48, missing=0
Y2: [0.30, 0.90], mean=0.74, missing=0
Y3: [0.20, 0.90], mean=0.37, missing=0
Y4: [1950.00, 32049.98], mean=6836.97, missing=0
Y5: [1.50, 14.49], mean=5.40, missing=0


In [5]:
missing_summary = pd.DataFrame({
    'Variable': df.columns,
    'Missing_Count': df.isna().sum().values,
    'Missing_Percent': (df.isna().sum().values / len(df) * 100).round(2)
}).sort_values('Missing_Count', ascending=False)

print("\nVariables with missing values:")
print(missing_summary[missing_summary['Missing_Count'] > 0].head(20))


Variables with missing values:
   Variable  Missing_Count  Missing_Percent
63      X51          72717            72.72
38      X54           3081             3.08
42      X60           3081             3.08
23      X21           3069             3.07
20      X16           3056             3.06
46      X15           3055             3.06
47      X20           3055             3.06
29      X41           3053             3.05
43      X14           3050             3.05
59      X36           3050             3.05
61      X40           3047             3.05
51      X26           3046             3.05
66      X57           3043             3.04
27      X38           3043             3.04
19      X13           3042             3.04
44      X18           3041             3.04
41      X59           3038             3.04
55      X30           3031             3.03
49      X24           3031             3.03
58      X35           3029             3.03


## SECTION 2: Feature Engineering

In [6]:
df_work = df.copy()

print("BMI Processing:")

if 'X29' in df_work.columns and df_work['X29'].isna().any():
    median_height = df_work['X29'].median()
    df_work['X29'].fillna(median_height, inplace=True)
    print(f"  Imputed X29 (height) with median: {median_height:.1f} cm")

if 'X30' in df_work.columns and df_work['X30'].isna().any():
    median_weight = df_work['X30'].median()
    df_work['X30'].fillna(median_weight, inplace=True)
    print(f"  Imputed X30 (weight) with median: {median_weight:.1f} kg")

if 'X31' in df_work.columns:
    df_work['BMI_IS_MISSING'] = df_work['X31'].isna().astype(int)
    
    if 'X29' in df_work.columns and 'X30' in df_work.columns:
        mask_missing_bmi = df_work['X31'].isna()
        mask_has_hw = df_work['X29'].notna() & df_work['X30'].notna()
        
        computed_bmi = df_work.loc[mask_missing_bmi & mask_has_hw, 'X30'] / \
                      (df_work.loc[mask_missing_bmi & mask_has_hw, 'X29'] / 100) ** 2
        
        df_work.loc[mask_missing_bmi & mask_has_hw, 'X31'] = computed_bmi
        
        df_work['BMI_WAS_COMPUTED'] = 0
        df_work.loc[mask_missing_bmi & mask_has_hw, 'BMI_WAS_COMPUTED'] = 1
        
        print(f"  Computed BMI for {(mask_missing_bmi & mask_has_hw).sum()} patients")
    
    if df_work['X31'].isna().any():
        median_bmi = df_work['X31'].median()
        df_work['X31'].fillna(median_bmi, inplace=True)
        print(f"  Filled remaining missing BMI with median: {median_bmi:.2f}")
    
    if 'X29' in df_work.columns:
        df_work.drop('X29', axis=1, inplace=True)
    if 'X30' in df_work.columns:
        df_work.drop('X30', axis=1, inplace=True)
    
    print(f"  Dropped X29 (height) and X30 (weight)")
    print(f"  Final BMI missing: {df_work['X31'].isna().sum()}")
else:
    print("  X31 (BMI) not found in dataset")

BMI Processing:
  Imputed X29 (height) with median: 168.0 cm
  Imputed X30 (weight) with median: 122.0 kg
  Computed BMI for 2985 patients
  Dropped X29 (height) and X30 (weight)
  Final BMI missing: 0


In [7]:
if 'X51' in df_work.columns:
    df_work['X51_IS_MISSING'] = df_work['X51'].isna().astype(int)
    df_work['X51'].fillna(-1, inplace=True)
    print(f"X51 missing values handled: {df_work['X51_IS_MISSING'].sum()} marked")

X51 missing values handled: 72717 marked


In [8]:
temporal_cols = ['X14', 'X18']

for col in temporal_cols:
    if col in df_work.columns:
        df_work[f'{col}_dt'] = pd.to_datetime(df_work[col], errors='coerce')
        
        df_work[f'{col}_MONTH'] = df_work[f'{col}_dt'].dt.month
        df_work[f'{col}_DOW'] = df_work[f'{col}_dt'].dt.dayofweek
        df_work[f'{col}_WEEKEND'] = (df_work[f'{col}_dt'].dt.dayofweek >= 5).astype(int)
        df_work[f'{col}_IS_MISSING'] = df_work[f'{col}_dt'].isna().astype(int)
        
        for feat in [f'{col}_MONTH', f'{col}_DOW', f'{col}_WEEKEND']:
            if df_work[feat].isna().any():
                df_work[feat].fillna(df_work[feat].median(), inplace=True)
        
        df_work.drop([col, f'{col}_dt'], axis=1, inplace=True)
        
        print(f"Processed temporal variable: {col}")

Processed temporal variable: X14
Processed temporal variable: X18


In [ ]:
if 'CHK_HOUR' in df_work.columns:
    df_work['CHK_HOUR'] = pd.to_numeric(df_work['CHK_HOUR'], errors='coerce')
    df_work['CHK_HOUR_IS_MISSING'] = df_work['CHK_HOUR'].isna().astype(int)

    if df_work['CHK_HOUR'].isna().any():
        median_hour = df_work['CHK_HOUR'].median()
        df_work['CHK_HOUR'] = df_work['CHK_HOUR'].fillna(median_hour)
    else:
        median_hour = None  

    df_work['CHK_HOUR'] = (df_work['CHK_HOUR'] % 24).astype(float)

    angle = 2 * np.pi * df_work['CHK_HOUR'] / 24.0
    df_work['CHK_HOUR_SIN'] = np.sin(angle)
    df_work['CHK_HOUR_COS'] = np.cos(angle)

    df_work = df_work.drop(columns=['CHK_HOUR'])


CHK_HOUR processed: impute -> cyclic encode (sin/cos) -> drop raw


## SECTION 3: Feature Filtering (Dictionary-Based + Uplift-Aligned)

In [10]:
feature_roles = {}

POST_TREATMENT_PREFIXES = ('X50','X51','X52','X15','X35','X40','X62')

for _, row in data_dict.iterrows():
    var_full = str(row['Variable']).strip()
    dict_cat = str(row.get('Category', '')).strip().lower()

    if var_full in outcome_cols:
        role = 'Outcome'
    elif var_full == treatment_col:
        role = 'Target'
    elif var_full.startswith(POST_TREATMENT_PREFIXES):
        role = 'Post-Treatment'
    elif dict_cat in ['administrative', 'encounter']:
        role = 'Administrative'
    else:
        role = 'Clinical'

    feature_roles[var_full] = role

    short = var_full.split('_')[0]   
    if short != var_full:
        feature_roles[short] = role

for col in df_work.columns:
    if col not in feature_roles:
        col_u = str(col).upper()

        if col == 'PAT_ID':
            feature_roles[col] = 'ID'
        elif col in outcome_cols:
            feature_roles[col] = 'Outcome'
        elif col == treatment_col:
            feature_roles[col] = 'Target'
        elif any(x in col_u for x in ['IS_MISSING', 'WAS_COMPUTED', '_SIN', '_COS']):
            feature_roles[col] = 'Clinical'  
        else:
            feature_roles[col] = 'Clinical'  

policy_table = []
for col in df_work.columns:
    role = feature_roles.get(col, 'Unknown')

    use_outcome = (role == 'Clinical')
    use_propensity = (role in ['Clinical', 'Administrative'])

    if role == 'Outcome':
        reason = 'Outcome variable - excluded'
    elif role == 'Target':
        reason = 'Treatment variable - excluded'
    elif role == 'ID':
        reason = 'Identifier - excluded'
    elif role == 'Post-Treatment':
        reason = 'Post-treatment variable - excluded'
    elif role == 'Clinical':
        reason = 'Clinical baseline feature'
    elif role == 'Administrative':
        reason = 'Administrative/encounter metadata - propensity only'
    else:
        reason = 'Uncategorized'

    policy_table.append({
        'Feature': col,
        'Category': role,  
        'Used_in_Outcome': use_outcome,
        'Used_in_Propensity': use_propensity,
        'Reason': reason
    })

policy_df = pd.DataFrame(policy_table)

print("\nFeature categorization summary:")
print(policy_df['Category'].value_counts())
print(f"\nFeatures for outcome models: {policy_df['Used_in_Outcome'].sum()}")
print(f"Features for propensity model: {policy_df['Used_in_Propensity'].sum()}")

print("\nPost-Treatment features found:")
print(policy_df[policy_df['Category'] == 'Post-Treatment'].to_string(index=False))

print("\nSample categorization:")
print(policy_df.head(50).to_string(index=False))



Feature categorization summary:
Category
Clinical          42
Administrative    24
Post-Treatment     7
Outcome            5
Target             1
Name: count, dtype: int64

Features for outcome models: 42
Features for propensity model: 66

Post-Treatment features found:
Feature       Category  Used_in_Outcome  Used_in_Propensity                             Reason
    X50 Post-Treatment            False               False Post-treatment variable - excluded
    X15 Post-Treatment            False               False Post-treatment variable - excluded
    X35 Post-Treatment            False               False Post-treatment variable - excluded
    X40 Post-Treatment            False               False Post-treatment variable - excluded
    X51 Post-Treatment            False               False Post-treatment variable - excluded
    X52 Post-Treatment            False               False Post-treatment variable - excluded
    X62 Post-Treatment            False               False Pos

In [11]:
outcome_features = policy_df[policy_df['Used_in_Outcome']]['Feature'].tolist()
propensity_features = policy_df[policy_df['Used_in_Propensity']]['Feature'].tolist()

print(f"\nOutcome model features: {len(outcome_features)}")
print(f"Propensity model features: {len(propensity_features)}")


Outcome model features: 42
Propensity model features: 66


In [12]:
numeric_features = []
for col in outcome_features:
    if col in df_work.columns:
        if pd.api.types.is_numeric_dtype(df_work[col]):
            numeric_features.append(col)

print(f"\nNumeric features for screening: {len(numeric_features)}")

def compute_f_filter(X, y, treatment, feature_name, order=2):
    """
    Compute F-statistic for treatment-feature interaction.
    
    Full model: y ~ T + X + X^2 + T:X + T:X^2
    Reduced model: y ~ T + X + X^2
    """
    try:
        mask = ~(np.isnan(X) | np.isnan(y) | np.isnan(treatment))
        X_clean = X[mask]
        y_clean = y[mask]
        T_clean = treatment[mask]
        
        if len(X_clean) < 50:  
            return np.nan, np.nan
        
   
        X_std = (X_clean - X_clean.mean()) / (X_clean.std() + 1e-8)
        
 
        features_reduced = [T_clean, X_std]
        if order >= 2:
            features_reduced.append(X_std ** 2)
        X_reduced = np.column_stack(features_reduced)
        
        features_full = features_reduced.copy()
        features_full.append(T_clean * X_std)  
        if order >= 2:
            features_full.append(T_clean * X_std ** 2)  
        X_full = np.column_stack(features_full)
        
        
        from sklearn.linear_model import LinearRegression
        
        model_reduced = LinearRegression().fit(X_reduced, y_clean)
        model_full = LinearRegression().fit(X_full, y_clean)
        
        rss_reduced = np.sum((y_clean - model_reduced.predict(X_reduced)) ** 2)
        rss_full = np.sum((y_clean - model_full.predict(X_full)) ** 2)
        
        df_reduced = X_reduced.shape[1]
        df_full = X_full.shape[1]
        df_diff = df_full - df_reduced
        n = len(y_clean)
        
        f_stat = ((rss_reduced - rss_full) / df_diff) / (rss_full / (n - df_full))
        p_value = 1 - stats.f.cdf(f_stat, df_diff, n - df_full)
        
        return f_stat, p_value
    except:
        return np.nan, np.nan

f_filter_results = []

T = df_work[treatment_col].values

for outcome in outcome_cols:
    if outcome not in df_work.columns:
        continue

    if outcome == 'Y4':
        y = np.log1p(df_work[outcome].values)
    else:
        y = df_work[outcome].values
    
    print(f"\nProcessing {outcome}...")
    
    for feat in numeric_features:
        X = df_work[feat].values
        f_stat, p_val = compute_f_filter(X, y, T, feat, order=2)
        
        f_filter_results.append({
            'Outcome': outcome,
            'Feature': feat,
            'F_Statistic': f_stat,
            'P_Value': p_val
        })

f_filter_df = pd.DataFrame(f_filter_results)
print("\nF-filter screening complete")


Numeric features for screening: 37

Processing Y1...

Processing Y2...

Processing Y3...

Processing Y4...

Processing Y5...

F-filter screening complete


In [13]:
feature_ranking = f_filter_df.groupby('Feature').agg({
    'F_Statistic': 'mean',
    'P_Value': lambda x: (x < 0.01).sum()
}).reset_index()

feature_ranking.columns = ['Feature', 'Mean_F_Stat', 'N_Significant']
feature_ranking = feature_ranking.sort_values(['N_Significant', 'Mean_F_Stat'], ascending=False)

print("\nTop 30 features by uplift signal:")
print(feature_ranking.head(30).to_string(index=False))

K = min(40, len(feature_ranking))
selected_numeric_features = feature_ranking.head(K)['Feature'].tolist()

critical_confounders = ['X1', 'X31', 'X2'] 
for confounder in critical_confounders:
    if confounder in numeric_features and confounder not in selected_numeric_features:
        selected_numeric_features.append(confounder)

print(f"\nSelected {len(selected_numeric_features)} numeric features after F-filter")


Top 30 features by uplift signal:
       Feature  Mean_F_Stat  N_Significant
            X8  6853.369329              5
            X7  5527.870579              5
            X6  4596.009089              5
            X5  2923.179041              5
            X4  1347.760076              4
            X3   738.165858              4
            X9   382.959312              4
            X2   290.101399              4
            X1    65.663053              4
           X26     3.393170              1
           X27     3.162285              1
           X31     2.406412              1
  CHK_HOUR_COS     2.031269              1
           X36     1.478662              1
X51_IS_MISSING     1.102887              1
           X32     1.950571              0
           X37     1.816235              0
           X42     1.665326              0
           X24     1.507039              0
           X23     1.464074              0
     X18_MONTH     1.410541              0
           X10     

In [14]:
print("\nFinal selected features for outcome models:")
print(selected_numeric_features)


Final selected features for outcome models:
['X8', 'X7', 'X6', 'X5', 'X4', 'X3', 'X9', 'X2', 'X1', 'X26', 'X27', 'X31', 'CHK_HOUR_COS', 'X36', 'X51_IS_MISSING', 'X32', 'X37', 'X42', 'X24', 'X23', 'X18_MONTH', 'X10', 'X14_DOW', 'X14_MONTH', 'X18_DOW', 'CHK_HOUR_SIN', 'X25', 'X11', 'X28', 'X14_IS_MISSING', 'BMI_IS_MISSING', 'BMI_WAS_COMPUTED', 'X18_IS_MISSING', 'X12', 'X18_WEEKEND', 'X14_WEEKEND', 'CHK_HOUR_IS_MISSING']


In [15]:
categorical_features = []
for col in outcome_features:
    if col in df_work.columns and col not in numeric_features:
        categorical_features.append(col)
        print(col)

final_features = selected_numeric_features + categorical_features

print(f"\nFinal feature set:")
print(f"  Numeric: {len(selected_numeric_features)}")
print(f"  Categorical: {len(categorical_features)}")
print(f"  Total: {len(final_features)}")

X33
X34
X38
X39
X41

Final feature set:
  Numeric: 37
  Categorical: 5
  Total: 42


## SECTION 4: Preprocessing Pipeline

In [16]:
final_numeric = [f for f in final_features if f in selected_numeric_features]
final_categorical = [f for f in final_features if f in categorical_features]

print(f"Final numeric features: {len(final_numeric)}")
print(f"Final categorical features: {len(final_categorical)}")

numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='constant', fill_value='Unknown')),
    ('onehot', OneHotEncoder(handle_unknown='ignore', min_frequency=50, sparse_output=False))
])

preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, final_numeric),
        ('cat', categorical_transformer, final_categorical)
    ],
    remainder='drop'
)


Final numeric features: 37
Final categorical features: 5


## SECTION 5: Cross-Validation Setup

In [17]:
n_folds = 5
cv = StratifiedKFold(n_splits=n_folds, shuffle=True, random_state=42)

print(f"Cross-validation: {n_folds}-fold stratified by treatment")

for fold_idx, (train_idx, test_idx) in enumerate(cv.split(df_work, df_work[treatment_col])):
    train_treatment_rate = df_work.iloc[train_idx][treatment_col].mean()
    test_treatment_rate = df_work.iloc[test_idx][treatment_col].mean()
    print(f"Fold {fold_idx + 1}: Train T=1 rate: {train_treatment_rate:.3f}, Test T=1 rate: {test_treatment_rate:.3f}")

Cross-validation: 5-fold stratified by treatment
Fold 1: Train T=1 rate: 0.236, Test T=1 rate: 0.236
Fold 2: Train T=1 rate: 0.236, Test T=1 rate: 0.236
Fold 3: Train T=1 rate: 0.236, Test T=1 rate: 0.236
Fold 4: Train T=1 rate: 0.236, Test T=1 rate: 0.236
Fold 5: Train T=1 rate: 0.236, Test T=1 rate: 0.236


## SECTION 6: Propensity Score Model

In [18]:
propensity_numeric = [f for f in propensity_features if f in df_work.columns and pd.api.types.is_numeric_dtype(df_work[f])]
propensity_categorical = [f for f in propensity_features if f in df_work.columns and f not in propensity_numeric]

prop_preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, propensity_numeric),
        ('cat', categorical_transformer, propensity_categorical)
    ],
    remainder='drop'
)

propensity_scores = np.zeros(len(df_work))
propensity_aucs = []

for fold_idx, (train_idx, test_idx) in enumerate(cv.split(df_work, df_work[treatment_col])):
    X_train = df_work.iloc[train_idx][propensity_features]
    y_train = df_work.iloc[train_idx][treatment_col]
    X_test = df_work.iloc[test_idx][propensity_features]
    y_test = df_work.iloc[test_idx][treatment_col]
    
    prop_pipeline = Pipeline([
        ('preprocess', prop_preprocessor),
        ('classifier', LogisticRegression(max_iter=1000, random_state=42))
    ])
    
    prop_pipeline.fit(X_train, y_train)

    propensity_scores[test_idx] = prop_pipeline.predict_proba(X_test)[:, 1]
    
    auc = roc_auc_score(y_test, propensity_scores[test_idx])
    propensity_aucs.append(auc)
    print(f"Fold {fold_idx + 1} Propensity AUC: {auc:.4f}")

print(f"\nMean Propensity AUC: {np.mean(propensity_aucs):.4f} ± {np.std(propensity_aucs):.4f}")

df_work['propensity_score'] = propensity_scores

Fold 1 Propensity AUC: 0.7402
Fold 2 Propensity AUC: 0.7391
Fold 3 Propensity AUC: 0.7433
Fold 4 Propensity AUC: 0.7374
Fold 5 Propensity AUC: 0.7405

Mean Propensity AUC: 0.7401 ± 0.0019


In [19]:
treatment_prevalence = df_work[treatment_col].mean()

ps_clipped = np.clip(propensity_scores, 0.01, 0.99)

ipw = np.where(
    df_work[treatment_col] == 1,
    treatment_prevalence / ps_clipped,
    (1 - treatment_prevalence) / (1 - ps_clipped)
)

weight_lower = np.percentile(ipw, 1)
weight_upper = np.percentile(ipw, 99)
ipw_clipped = np.clip(ipw, weight_lower, weight_upper)

df_work['ipw'] = ipw_clipped

ess = (ipw_clipped.sum() ** 2) / (np.sum(ipw_clipped ** 2))

print(f"  Mean: {ipw_clipped.mean():.3f}")
print(f"  Median: {np.median(ipw_clipped):.3f}")
print(f"  Min: {ipw_clipped.min():.3f}")
print(f"  Max: {ipw_clipped.max():.3f}")
print(f"  Std: {ipw_clipped.std():.3f}")
print(f"  ESS: {ess:.1f} ({ess/len(ipw_clipped)*100:.1f}% of N)")

  Mean: 0.996
  Median: 0.897
  Min: 0.349
  Max: 2.913
  Std: 0.399
  ESS: 86167.6 (86.2% of N)


### Covariate balance diagnostics (SMD before vs after IPW)

In [20]:
def _weighted_mean(x, w):
    return np.sum(w * x) / np.sum(w)

def _weighted_var(x, w):
    mu = _weighted_mean(x, w)
    return np.sum(w * (x - mu) ** 2) / np.sum(w)

def compute_smd(x, t, w=None):
    x = np.asarray(x, dtype=float)
    t = np.asarray(t).astype(int)
    mask1 = t == 1
    mask0 = t == 0
    if w is None:
        m1, m0 = np.nanmean(x[mask1]), np.nanmean(x[mask0])
        v1, v0 = np.nanvar(x[mask1], ddof=0), np.nanvar(x[mask0], ddof=0)
    else:
        w = np.asarray(w, dtype=float)
        m1, m0 = _weighted_mean(x[mask1], w[mask1]), _weighted_mean(x[mask0], w[mask0])
        v1, v0 = _weighted_var(x[mask1], w[mask1]), _weighted_var(x[mask0], w[mask0])
    denom = np.sqrt((v1 + v0) / 2.0) + 1e-12
    return (m1 - m0) / denom

baseline_clinical_features = [f for f in final_features if feature_roles.get(f, '') == 'Clinical']

baseline_clinical_numeric = [
    f for f in baseline_clinical_features
    if f in df_work.columns and pd.api.types.is_numeric_dtype(df_work[f])
]

print(f"Baseline clinical numeric features for balance: {len(baseline_clinical_numeric)}")

balance_rows = []
for f in baseline_clinical_numeric:
    x = df_work[f].values
    t = df_work[treatment_col].values
    smd_before = compute_smd(x, t, w=None)
    smd_after = compute_smd(x, t, w=df_work['ipw'].values)
    balance_rows.append((f, smd_before, smd_after))

balance_df = pd.DataFrame(balance_rows, columns=['feature', 'SMD_before', 'SMD_after_IPW'])
balance_df['abs_SMD_after'] = balance_df['SMD_after_IPW'].abs()
pct_balanced = (balance_df['abs_SMD_after'] < 0.1).mean() * 100.0

print(f"% features with |SMD| < 0.1 after IPW: {pct_balanced:.1f}%")

display(balance_df.sort_values('abs_SMD_after', ascending=False).head(20))


Baseline clinical numeric features for balance: 37
% features with |SMD| < 0.1 after IPW: 73.0%


,feature,SMD_before,SMD_after_IPW,abs_SMD_after
3,X5,0.547544,0.093674,0.093674
4,X4,0.441309,0.083011,0.083011
1,X7,0.212138,0.073950,0.073950
5,X3,0.288733,0.053405,0.053405
2,X6,0.101462,0.046873,0.046873
7,X2,0.139977,0.028283,0.028283
0,X8,-0.325933,-0.021363,0.021363
8,X1,0.071462,0.009778,0.009778
6,X9,-0.153450,-0.009772,0.009772
27,X11,0.005719,0.008178,0.008178


## SECTION 7: Causal Models - Meta-Learners

In [21]:
base_models = {
    'RandomForest': RandomForestRegressor(n_estimators=100, max_depth=10, min_samples_leaf=20, random_state=42, n_jobs=-1),
    'GradientBoosting': GradientBoostingRegressor(n_estimators=100, max_depth=5, learning_rate=0.1, random_state=42),
    'HistGradientBoosting': HistGradientBoostingRegressor(max_iter=100, max_depth=5, learning_rate=0.1, random_state=42)
}

if HAS_XGB:
    base_models['XGBoost'] = xgb.XGBRegressor(n_estimators=100, max_depth=5, learning_rate=0.1, random_state=42, n_jobs=-1)

if HAS_LGB:
    base_models['LightGBM'] = lgb.LGBMRegressor(n_estimators=100, max_depth=5, learning_rate=0.1, random_state=42, verbose=-1)

if HAS_CB:
    base_models['CatBoost'] = cb.CatBoostRegressor(iterations=100, depth=5, learning_rate=0.1, random_state=42, verbose=0)

print(f"Base models: {list(base_models.keys())}")

Base models: ['RandomForest', 'GradientBoosting', 'HistGradientBoosting', 'XGBoost', 'LightGBM', 'CatBoost']


In [22]:
class XLearner:
    def __init__(self, model_t1, model_t0, model_te1, model_te0):
        self.model_t1 = model_t1  
        self.model_t0 = model_t0  
        self.model_te1 = model_te1  
        self.model_te0 = model_te0  
    
    def fit(self, X, y, treatment, sample_weight=None):
        mask_t1 = treatment == 1
        mask_t0 = treatment == 0
        
        X_t1 = X[mask_t1]
        y_t1 = y[mask_t1]
        X_t0 = X[mask_t0]
        y_t0 = y[mask_t0]
        
        w_t1 = sample_weight[mask_t1] if sample_weight is not None else None
        w_t0 = sample_weight[mask_t0] if sample_weight is not None else None
        
        self.model_t1.fit(X_t1, y_t1, sample_weight=w_t1)
        self.model_t0.fit(X_t0, y_t0, sample_weight=w_t0)
        
       
        te_imputed_t1 = y_t1 - self.model_t0.predict(X_t1)
        te_imputed_t0 = self.model_t1.predict(X_t0) - y_t0
        

        self.model_te1.fit(X_t1, te_imputed_t1, sample_weight=w_t1)
        self.model_te0.fit(X_t0, te_imputed_t0, sample_weight=w_t0)
        
        return self
    
    def predict(self, X, propensity=None):
        te1 = self.model_te1.predict(X)
        te0 = self.model_te0.predict(X)
        
        if propensity is not None:            
            te = propensity * te0 + (1 - propensity) * te1
        else:     
            te = (te0 + te1) / 2
        
        return te
    
    def predict_counterfactuals(self, X):
        mu1 = self.model_t1.predict(X)
        mu0 = self.model_t0.predict(X)
        return mu1, mu0

In [23]:
class TLearner:
    def __init__(self, model_t1, model_t0):
        self.model_t1 = model_t1
        self.model_t0 = model_t0
    
    def fit(self, X, y, treatment, sample_weight=None):
        mask_t1 = treatment == 1
        mask_t0 = treatment == 0
        
        X_t1 = X[mask_t1]
        y_t1 = y[mask_t1]
        X_t0 = X[mask_t0]
        y_t0 = y[mask_t0]
        
        w_t1 = sample_weight[mask_t1] if sample_weight is not None else None
        w_t0 = sample_weight[mask_t0] if sample_weight is not None else None
        
        self.model_t1.fit(X_t1, y_t1, sample_weight=w_t1)
        self.model_t0.fit(X_t0, y_t0, sample_weight=w_t0)
        
        return self
    
    def predict(self, X):
        mu1 = self.model_t1.predict(X)
        mu0 = self.model_t0.predict(X)
        return mu1 - mu0
    
    def predict_counterfactuals(self, X):
        mu1 = self.model_t1.predict(X)
        mu0 = self.model_t0.predict(X)
        return mu1, mu0

In [24]:
class SLearner:
    def __init__(self, model):
        self.model = model

    def fit(self, X, y, treatment, sample_weight=None):
        Xt = np.column_stack([X, treatment.reshape(-1,1)])
        if sample_weight is None:
            self.model.fit(Xt, y)
        else:
            self.model.fit(Xt, y, sample_weight=sample_weight)
        return self

    def predict_counterfactuals(self, X):
        n = X.shape[0]
        X1 = np.column_stack([X, np.ones((n,1))])
        X0 = np.column_stack([X, np.zeros((n,1))])
        mu1 = self.model.predict(X1)
        mu0 = self.model.predict(X0)
        return mu1, mu0

    def predict(self, X):
        mu1, mu0 = self.predict_counterfactuals(X)
        return mu1 - mu0

class DRLearner:
    def __init__(self, model_mu1, model_mu0, model_tau):
        self.model_mu1 = model_mu1
        self.model_mu0 = model_mu0
        self.model_tau = model_tau

    def fit(self, X, y, treatment, propensity, sample_weight=None):
        mask1 = treatment == 1
        mask0 = treatment == 0

        if sample_weight is None:
            self.model_mu1.fit(X[mask1], y[mask1])
            self.model_mu0.fit(X[mask0], y[mask0])
        else:
            self.model_mu1.fit(X[mask1], y[mask1], sample_weight=sample_weight[mask1])
            self.model_mu0.fit(X[mask0], y[mask0], sample_weight=sample_weight[mask0])

        mu1 = self.model_mu1.predict(X)
        mu0 = self.model_mu0.predict(X)

        e = np.clip(propensity, 1e-3, 1-1e-3)
        t = treatment.astype(float)

        phi = (t * (y - mu1) / e) - ((1 - t) * (y - mu0) / (1 - e)) + (mu1 - mu0)

        if sample_weight is None:
            self.model_tau.fit(X, phi)
        else:
            self.model_tau.fit(X, phi, sample_weight=sample_weight)

        return self

    def predict(self, X):
        return self.model_tau.predict(X)

    def predict_counterfactuals(self, X):
        mu1 = self.model_mu1.predict(X)
        mu0 = self.model_mu0.predict(X)
        return mu1, mu0


class MLPDropoutRegressor:
    def __init__(self, input_dim=None, hidden=(256,128), dropout=0.2, lr=1e-3, 
                 batch_size=1024, epochs=20, weight_decay=1e-5, random_state=42, device=None):
        self.input_dim = input_dim
        self.hidden = hidden
        self.dropout = dropout
        self.lr = lr
        self.batch_size = batch_size
        self.epochs = epochs
        self.weight_decay = weight_decay
        self.random_state = random_state
        self.device = device

    def _build(self, d):
        layers = []
        prev = d
        for h in self.hidden:
            layers += [nn.Linear(prev, h), nn.ReLU(), nn.Dropout(self.dropout)]
            prev = h
        layers += [nn.Linear(prev, 1)]
        return nn.Sequential(*layers)

    def fit(self, X, y, sample_weight=None):
        if not HAS_TORCH:
            raise RuntimeError("PyTorch is required for MLPDropoutRegressor.")
        X = np.asarray(X, dtype=np.float32)
        y = np.asarray(y, dtype=np.float32).reshape(-1,1)
        n, d = X.shape
        self.input_dim = d

        torch.manual_seed(self.random_state)
        device = self.device or ("cuda" if torch.cuda.is_available() else "cpu")
        self._device = device

        self.net_ = self._build(d).to(device)
        opt = optim.AdamW(self.net_.parameters(), lr=self.lr, weight_decay=self.weight_decay)
        loss_fn = nn.MSELoss(reduction='none')

        idx = np.arange(n)
        for ep in range(self.epochs):
            np.random.shuffle(idx)
            self.net_.train()
            for start in range(0, n, self.batch_size):
                batch = idx[start:start+self.batch_size]
                xb = torch.from_numpy(X[batch]).to(device)
                yb = torch.from_numpy(y[batch]).to(device)
                pred = self.net_(xb)
                l = loss_fn(pred, yb).squeeze(-1)
                if sample_weight is not None:
                    wb = torch.from_numpy(np.asarray(sample_weight[batch], dtype=np.float32)).to(device)
                    l = l * wb
                l = l.mean()
                opt.zero_grad()
                l.backward()
                opt.step()
        return self

    def predict(self, X):
        X = np.asarray(X, dtype=np.float32)
        device = getattr(self, "_device", "cpu")
        self.net_.eval()
        with torch.no_grad():
            xb = torch.from_numpy(X).to(device)
            pred = self.net_(xb).cpu().numpy().reshape(-1)
        return pred


In [25]:

cv_splits = list(cv.split(df_work, df_work[treatment_col]))

results = defaultdict(lambda: defaultdict(dict))

X_all_raw = df_work[final_features].copy()
T_all = df_work[treatment_col].values.astype(int)
ps_all = df_work['propensity_score'].values
ipw_all = df_work['ipw'].values

preprocessor.fit(X_all_raw)
X_all = preprocessor.transform(X_all_raw)

default_tree = 'XGBoost'

def inverse_if_y4(outcome, arr_log):
    if outcome == 'Y4':
        return expm1(arr_log)
    return arr_log

def compute_r_loss(y_test, mu_test, te_test, T_test, ps_test):
    y_res = y_test - mu_test
    t_res = T_test - ps_test
    eps = 1e-3
    t_res = np.clip(t_res, -1 + eps, 1 - eps)
    return float(np.mean((y_res - te_test * t_res) ** 2))

def compute_factual_metrics(y_true, mu1_pred, mu0_pred, T, outcome):
    y_true_orig = inverse_if_y4(outcome, y_true)
    mu1_orig = inverse_if_y4(outcome, mu1_pred)
    mu0_orig = inverse_if_y4(outcome, mu0_pred)

    out = {}
    mask1 = T == 1
    mask0 = T == 0
    if mask1.sum() > 0:
        out['T=1'] = {
            'MAE': float(mean_absolute_error(y_true_orig[mask1], mu1_orig[mask1])),
            'RMSE': float(np.sqrt(mean_squared_error(y_true_orig[mask1], mu1_orig[mask1])))
        }
    if mask0.sum() > 0:
        out['T=0'] = {
            'MAE': float(mean_absolute_error(y_true_orig[mask0], mu0_orig[mask0])),
            'RMSE': float(np.sqrt(mean_squared_error(y_true_orig[mask0], mu0_orig[mask0])))
        }
    return out

def build_estimators(model_name):
    """Return a dict of TE estimators (at least 4) using the given base model."""
    base = base_models[model_name]
    est = {}

    est['S-learner'] = SLearner(clone(base))
    est['T-learner'] = TLearner(clone(base), clone(base))
    est['X-learner'] = XLearner(clone(base), clone(base), clone(base), clone(base))
    est['DR-learner'] = DRLearner(clone(base), clone(base), clone(base))

    if 'HAS_TORCH' in globals() and HAS_TORCH:
        est['T-learner-MLPDropout'] = 'MLP_DROPOUT_TLEARNER'

    return est


for outcome in outcome_cols:
    if outcome not in df_work.columns:
        print(f"\nSkipping {outcome} (not in dataset)")
        continue

    if outcome == 'Y4':
        y_all = np.log1p(df_work[outcome].values) 
    else:
        y_all = df_work[outcome].values.astype(float)

    estimators = build_estimators(default_tree)

    for est_name, est_obj in estimators.items():
        print(f"\n-- Estimator: {est_name} (base={default_tree})")

        te_oof = np.zeros(len(df_work), dtype=float)
        mu1_oof = np.zeros(len(df_work), dtype=float)
        mu0_oof = np.zeros(len(df_work), dtype=float)

        rloss_folds = []

        for fold_idx, (train_idx, test_idx) in enumerate(cv_splits):
            X_train = X_all[train_idx]
            X_test = X_all[test_idx]
            y_train = y_all[train_idx]
            y_test = y_all[test_idx]
            T_train = T_all[train_idx]
            T_test = T_all[test_idx]
            ps_test = ps_all[test_idx]
            ipw_train = ipw_all[train_idx]

            if est_name == 'T-learner-MLPDropout':
                d_in = X_train.shape[1]
                mu1_model = MLPDropoutRegressor(input_dim=d_in, dropout=0.2, epochs=15, batch_size=2048)
                mu0_model = MLPDropoutRegressor(input_dim=d_in, dropout=0.2, epochs=15, batch_size=2048)
                tlearner = TLearner(mu1_model, mu0_model)
                tlearner.fit(X_train, y_train, T_train, sample_weight=ipw_train)
                te_test = tlearner.predict(X_test)
                mu1_test, mu0_test = tlearner.predict_counterfactuals(X_test)
                mu_model = None  
                mu_model = clone(base_models[default_tree])
                mu_model.fit(X_train, y_train)
                mu_test = mu_model.predict(X_test)
            else:
                if est_name == 'S-learner':
                    est_fold = SLearner(clone(base_models[default_tree]))
                    est_fold.fit(X_train, y_train, T_train, sample_weight=ipw_train)
                    te_test = est_fold.predict(X_test)
                    mu1_test, mu0_test = est_fold.predict_counterfactuals(X_test)
                elif est_name == 'T-learner':
                    est_fold = TLearner(clone(base_models[default_tree]), clone(base_models[default_tree]))
                    est_fold.fit(X_train, y_train, T_train, sample_weight=ipw_train)
                    te_test = est_fold.predict(X_test)
                    mu1_test, mu0_test = est_fold.predict_counterfactuals(X_test)
                elif est_name == 'X-learner':
                    est_fold = XLearner(clone(base_models[default_tree]), clone(base_models[default_tree]),
                                        clone(base_models[default_tree]), clone(base_models[default_tree]))
                    est_fold.fit(X_train, y_train, T_train, sample_weight=ipw_train)
                    te_test = est_fold.predict(X_test, propensity=ps_test)
                    mu1_test, mu0_test = est_fold.predict_counterfactuals(X_test)
                elif est_name == 'DR-learner':
                    est_fold = DRLearner(clone(base_models[default_tree]), clone(base_models[default_tree]),
                                         clone(base_models[default_tree]))
                    est_fold.fit(X_train, y_train, T_train, propensity=ps_all[train_idx], sample_weight=ipw_train)
                    te_test = est_fold.predict(X_test)
                    mu1_test, mu0_test = est_fold.predict_counterfactuals(X_test)
                else:
                    raise ValueError(f"Unknown estimator: {est_name}")

                mu_model = clone(base_models[default_tree])
                mu_model.fit(X_train, y_train)
                mu_test = mu_model.predict(X_test)

            rloss = compute_r_loss(y_test, mu_test, te_test, T_test, ps_test)
            rloss_folds.append(rloss)

            te_oof[test_idx] = te_test
            mu1_oof[test_idx] = mu1_test
            mu0_oof[test_idx] = mu0_test

            print(f"  Fold {fold_idx+1}/{n_folds}: R-loss={rloss:.6f}")

        rloss_mean = float(np.mean(rloss_folds))
        rloss_std = float(np.std(rloss_folds))

        factual = compute_factual_metrics(y_all, mu1_oof, mu0_oof, T_all, outcome)

        print(f"  R-loss (training scale): mean={rloss_mean:.6f} ± {rloss_std:.6f}")
        if outcome == 'Y4':
            print("  NOTE: R-loss above is on log1p(cost) scale (labeled).")
        for arm in factual:
            print(f"  Factual {arm} (ORIGINAL units): MAE={factual[arm]['MAE']:.4f}, RMSE={factual[arm]['RMSE']:.4f}")

        results[outcome][est_name] = {
            'te': te_oof,
            'mu1': mu1_oof,
            'mu0': mu0_oof,
            'R_loss_mean': rloss_mean,
            'R_loss_std': rloss_std,
            'R_loss_folds': rloss_folds,
            'factual_metrics_orig': factual,
            'training_scale': 'log1p' if outcome == 'Y4' else 'original'
        }

best_model_per_outcome = {}
for outcome in results:
    best_est = min(results[outcome].keys(), key=lambda k: results[outcome][k]['R_loss_mean'])
    best_model_per_outcome[outcome] = best_est

for outcome, best_est in best_model_per_outcome.items():
    r = results[outcome][best_est]['R_loss_mean']
    scale = results[outcome][best_est]['training_scale']
    print(f"{outcome}: {best_est} | R-loss={r:.6f} (scale={scale})")

all_predictions = defaultdict(lambda: defaultdict(dict))
r_loss_metrics = {}
factual_metrics = {}

for outcome in best_model_per_outcome:
    best_est = best_model_per_outcome[outcome]
    all_predictions[outcome]['te'] = results[outcome][best_est]['te']
    all_predictions[outcome]['mu1'] = results[outcome][best_est]['mu1']
    all_predictions[outcome]['mu0'] = results[outcome][best_est]['mu0']
    r_loss_metrics[outcome] = {
        'R_loss_mean': results[outcome][best_est]['R_loss_mean'],
        'R_loss_std': results[outcome][best_est]['R_loss_std'],
        'estimator': best_est,
        'scale': results[outcome][best_est]['training_scale']
    }
    factual_metrics[outcome] = results[outcome][best_est]['factual_metrics_orig']



-- Estimator: S-learner (base=XGBoost)
  Fold 1/5: R-loss=0.003835
  Fold 2/5: R-loss=0.003787
  Fold 3/5: R-loss=0.003746
  Fold 4/5: R-loss=0.003752
  Fold 5/5: R-loss=0.003801
  R-loss (training scale): mean=0.003784 ± 0.000033
  Factual T=1 (ORIGINAL units): MAE=0.0504, RMSE=0.0582
  Factual T=0 (ORIGINAL units): MAE=0.0501, RMSE=0.0578

-- Estimator: T-learner (base=XGBoost)
  Fold 1/5: R-loss=0.003843
  Fold 2/5: R-loss=0.003797
  Fold 3/5: R-loss=0.003752
  Fold 4/5: R-loss=0.003762
  Fold 5/5: R-loss=0.003812
  R-loss (training scale): mean=0.003793 ± 0.000033
  Factual T=1 (ORIGINAL units): MAE=0.0506, RMSE=0.0585
  Factual T=0 (ORIGINAL units): MAE=0.0501, RMSE=0.0579

-- Estimator: X-learner (base=XGBoost)
  Fold 1/5: R-loss=0.003842
  Fold 2/5: R-loss=0.003792
  Fold 3/5: R-loss=0.003749
  Fold 4/5: R-loss=0.003759
  Fold 5/5: R-loss=0.003807
  R-loss (training scale): mean=0.003790 ± 0.000034
  Factual T=1 (ORIGINAL units): MAE=0.0506, RMSE=0.0585
  Factual T=0 (ORIGINAL 

## SECTION 8: Model Evaluation (Causal-Safe Metrics)

In [26]:
rows = []
for outcome in results:
    for est in results[outcome]:
        rows.append({
            'Outcome': outcome,
            'Estimator': est,
            'R_loss_mean': results[outcome][est]['R_loss_mean'],
            'R_loss_std': results[outcome][est]['R_loss_std'],
            'TrainingScale': results[outcome][est]['training_scale']
        })
rloss_df = pd.DataFrame(rows).sort_values(['Outcome','R_loss_mean'])
display(rloss_df)

print("\nBEST ESTIMATOR PER OUTCOME (lowest mean R-loss):")
for outcome, best_est in best_model_per_outcome.items():
    r = results[outcome][best_est]['R_loss_mean']
    s = results[outcome][best_est]['training_scale']
    print(f"  {outcome}: {best_est} | R-loss={r:.6f} (scale={s})")


factual_rows = []
for outcome in best_model_per_outcome:
    best_est = best_model_per_outcome[outcome]
    fm = results[outcome][best_est]['factual_metrics_orig']
    for arm in fm:
        factual_rows.append({
            'Outcome': outcome,
            'SelectedEstimator': best_est,
            'Arm': arm,
            'MAE': fm[arm]['MAE'],
            'RMSE': fm[arm]['RMSE']
        })

factual_df = pd.DataFrame(factual_rows)
display(factual_df)


,Outcome,Estimator,R_loss_mean,R_loss_std,TrainingScale
0,Y1,S-learner,0.003784,0.000033,original
2,Y1,X-learner,0.003790,0.000034,original
1,Y1,T-learner,0.003793,0.000033,original
3,Y1,DR-learner,0.003830,0.000037,original
4,Y1,T-learner-MLPDropout,0.003903,0.000031,original
5,Y2,S-learner,0.003626,0.000040,original
7,Y2,X-learner,0.003632,0.000042,original
6,Y2,T-learner,0.003636,0.000041,original
8,Y2,DR-learner,0.003679,0.000042,original
9,Y2,T-learner-MLPDropout,0.003702,0.000034,original



BEST ESTIMATOR PER OUTCOME (lowest mean R-loss):
  Y1: S-learner | R-loss=0.003784 (scale=original)
  Y2: S-learner | R-loss=0.003626 (scale=original)
  Y3: S-learner | R-loss=0.003767 (scale=original)
  Y4: S-learner | R-loss=0.021625 (scale=log1p)
  Y5: S-learner | R-loss=2.238164 (scale=original)


,Outcome,SelectedEstimator,Arm,MAE,RMSE
0,Y1,S-learner,T=1,0.050378,0.058180
1,Y1,S-learner,T=0,0.050070,0.057844
2,Y2,S-learner,T=1,0.050310,0.058119
3,Y2,S-learner,T=0,0.050007,0.057801
4,Y3,S-learner,T=1,0.050039,0.057924
5,Y3,S-learner,T=0,0.049937,0.057731
6,Y4,S-learner,T=1,25.325226,29.446127
7,Y4,S-learner,T=0,25.031260,28.915939
8,Y5,S-learner,T=1,1.254098,1.449133
9,Y5,S-learner,T=0,1.256774,1.451441


In [27]:
for outcome in outcome_cols:
    if outcome not in all_predictions:
        continue
    
    te = all_predictions[outcome]['te']
    
    print(f"\n{outcome}:")
    print(f"  TE range: [{te.min():.4f}, {te.max():.4f}]")
    print(f"  TE mean: {te.mean():.4f}")
    print(f"  TE median: {np.median(te):.4f}")
    print(f"  TE std: {te.std():.4f}")
    
    if outcome == 'Y4':  
        print(f"  Expected: TE > 0 (surgery more expensive)")
        print(f"  Proportion TE > 0: {(te > 0).mean():.2%}")
    
    if outcome == 'Y5': 
        print(f"  Expected: TE > 0 (surgery longer recovery)")
        print(f"  Proportion TE > 0: {(te > 0).mean():.2%}")
    
    outliers = np.abs(te - te.mean()) > 3 * te.std()
    if outliers.sum() > 0:
        print(f"  WARNING: {outliers.sum()} outliers detected (>3 std)")


Y1:
  TE range: [-0.0075, 0.5322]
  TE mean: 0.2513
  TE median: 0.2562
  TE std: 0.1198

Y2:
  TE range: [-0.4200, -0.0032]
  TE mean: -0.2000
  TE median: -0.1978
  TE std: 0.0948

Y3:
  TE range: [-0.0142, 0.5263]
  TE mean: 0.2509
  TE median: 0.2431
  TE std: 0.1200

Y4:
  TE range: [1.7881, 2.7771]
  TE mean: 2.3695
  TE median: 2.3686
  TE std: 0.2373
  Expected: TE > 0 (surgery more expensive)
  Proportion TE > 0: 100.00%

Y5:
  TE range: [3.2346, 8.6972]
  TE mean: 6.0038
  TE median: 6.0279
  TE std: 0.9538
  Expected: TE > 0 (surgery longer recovery)
  Proportion TE > 0: 100.00%


## SECTION 9: Clinical Heterogeneity Drivers

In [28]:

baseline_clinical_features = [f for f in final_features if feature_roles.get(f, '') == 'Clinical']
print(f"Baseline clinical features eligible for explanation: {len(baseline_clinical_features)}")

X_full_raw = df_work[final_features].copy()
X_full = preprocessor.transform(X_full_raw)

def fit_best_estimator_full(outcome, est_name):
    if outcome == 'Y4':
        y = np.log1p(df_work[outcome].values)
    else:
        y = df_work[outcome].values.astype(float)

    base = base_models[default_tree]

    if est_name == 'S-learner':
        est = SLearner(clone(base))
        est.fit(X_full, y, T_all, sample_weight=ipw_all)
        return est
    if est_name == 'T-learner':
        est = TLearner(clone(base), clone(base))
        est.fit(X_full, y, T_all, sample_weight=ipw_all)
        return est
    if est_name == 'X-learner':
        est = XLearner(clone(base), clone(base), clone(base), clone(base))
        est.fit(X_full, y, T_all, sample_weight=ipw_all)
        return est
    if est_name == 'DR-learner':
        est = DRLearner(clone(base), clone(base), clone(base))
        est.fit(X_full, y, T_all, propensity=ps_all, sample_weight=ipw_all)
        return est
    if est_name == 'T-learner-MLPDropout':
        if not ('HAS_TORCH' in globals() and HAS_TORCH):
            raise RuntimeError("PyTorch not available for MLPDropout.")
        d_in = X_full.shape[1]
        mu1_model = MLPDropoutRegressor(input_dim=d_in, dropout=0.2, epochs=25, batch_size=4096)
        mu0_model = MLPDropoutRegressor(input_dim=d_in, dropout=0.2, epochs=25, batch_size=4096)
        est = TLearner(mu1_model, mu0_model)
        est.fit(X_full, y, T_all, sample_weight=ipw_all)
        return est
    raise ValueError(f"Unknown estimator: {est_name}")

def predict_te_full(est, X_raw_df, outcome):
    Xmat = preprocessor.transform(X_raw_df[final_features])
    if isinstance(est, XLearner):
        te = est.predict(Xmat, propensity=df_work['propensity_score'].values)
    else:
        te = est.predict(Xmat)
    if outcome == 'Y4':
        mu1_log, mu0_log = est.predict_counterfactuals(Xmat)
        te = expm1(mu1_log) - expm1(mu0_log)
    return te

driver_results = []
for outcome in best_model_per_outcome:
    best_est = best_model_per_outcome[outcome]
    print(f"\nOutcome {outcome}: explaining heterogeneity for selected model = {best_est}")

    est_full = fit_best_estimator_full(outcome, best_est)

    te_base = predict_te_full(est_full, X_full_raw, outcome)

    rng = np.random.default_rng(42)
    importances = []

    for f in baseline_clinical_features:
        if f not in X_full_raw.columns:
            continue
        deltas = []
        for r in range(5):
            Xp = X_full_raw.copy()
            Xp[f] = rng.permutation(Xp[f].values)
            te_perm = predict_te_full(est_full, Xp, outcome)
            deltas.append(np.mean(np.abs(te_perm - te_base)))
        imp = float(np.mean(deltas))
        importances.append((f, imp))

    imp_df = pd.DataFrame(importances, columns=['feature', 'perm_importance_mean_abs_delta_TE'])
    imp_df = imp_df.sort_values('perm_importance_mean_abs_delta_TE', ascending=False)

    topk = imp_df.head(15)
    display(topk)

    for rank, row in enumerate(topk.itertuples(index=False), start=1):
        driver_results.append({
            'Outcome': outcome,
            'SelectedEstimator': best_est,
            'Rank': rank,
            'Feature': row.feature,
            'Importance': row.perm_importance_mean_abs_delta_TE
        })

drivers_for_slides = pd.DataFrame(driver_results)
print("\nTop drivers table (for slides):")
display(drivers_for_slides.head(30))


Baseline clinical features eligible for explanation: 42

Outcome Y1: explaining heterogeneity for selected model = S-learner


,feature,perm_importance_mean_abs_delta_TE
3,X5,0.082104
4,X4,0.063464
5,X3,0.048614
7,X2,0.031620
8,X1,0.014760
12,CHK_HOUR_COS,0.001001
11,X31,0.000965
19,X23,0.000923
10,X27,0.000897
38,X34,0.000885



Outcome Y2: explaining heterogeneity for selected model = S-learner


,feature,perm_importance_mean_abs_delta_TE
2,X6,0.064145
3,X5,0.051841
4,X4,0.038040
5,X3,0.025120
7,X2,0.011656
11,X31,0.000960
26,X25,0.000924
38,X34,0.000825
12,CHK_HOUR_COS,0.000781
17,X42,0.000763



Outcome Y3: explaining heterogeneity for selected model = S-learner


...

,feature,perm_importance_mean_abs_delta_TE
0,X8,3335.801514
1,X7,2676.300049
2,X6,2005.732422
3,X5,1332.857910
4,X4,663.078674
10,X27,0.915813
11,X31,0.627856
25,CHK_HOUR_SIN,0.620492
38,X34,0.580709
26,X25,0.523824



Outcome Y5: explaining heterogeneity for selected model = S-learner


,feature,perm_importance_mean_abs_delta_TE
6,X9,0.651000
0,X8,0.499847
1,X7,0.382121
2,X6,0.258602
3,X5,0.109134
19,X23,0.036983
11,X31,0.034416
26,X25,0.031726
10,X27,0.028333
25,CHK_HOUR_SIN,0.027102



Top drivers table (for slides):


,Outcome,SelectedEstimator,Rank,Feature,Importance
0,Y1,S-learner,1,X5,0.082104
1,Y1,S-learner,2,X4,0.063464
2,Y1,S-learner,3,X3,0.048614
3,Y1,S-learner,4,X2,0.031620
4,Y1,S-learner,5,X1,0.014760
5,Y1,S-learner,6,CHK_HOUR_COS,0.001001
6,Y1,S-learner,7,X31,0.000965
7,Y1,S-learner,8,X23,0.000923
8,Y1,S-learner,9,X27,0.000897
9,Y1,S-learner,10,X34,0.000885


## SECTION 10: Preference Heterogeneity (CDS Layer)

In [29]:
benefits = {}

if 'Y1' in all_predictions:
    benefits['Pain_Reduction'] = all_predictions['Y1']['te']  

if 'Y2' in all_predictions:
    benefits['Short_Term_Pain'] = -all_predictions['Y2']['te']  

if 'Y3' in all_predictions:
    benefits['Functional_Improvement'] = all_predictions['Y3']['te']  

if 'Y4' in all_predictions:
    mu1_log = all_predictions['Y4']['mu1']
    mu0_log = all_predictions['Y4']['mu0']
    cost_diff = expm1(mu1_log) - expm1(mu0_log)  
    benefits['Cost_Savings'] = -cost_diff 

if 'Y5' in all_predictions:
    benefits['Rehab_Time_Reduction'] = -all_predictions['Y5']['te'] 

print(f"\nDefined {len(benefits)} benefit dimensions")

benefits_normalized = {}
for name, values in benefits.items():
    vmin, vmax = values.min(), values.max()
    if vmax > vmin:
        normalized = (values - vmin) / (vmax - vmin)
    else:
        normalized = np.ones_like(values) * 0.5
    benefits_normalized[name] = normalized
    print(f"  {name}: range=[{vmin:.4f}, {vmax:.4f}]")


Defined 5 benefit dimensions
  Pain_Reduction: range=[-0.0075, 0.5322]
  Short_Term_Pain: range=[0.0032, 0.4200]
  Functional_Improvement: range=[-0.0142, 0.5263]
  Cost_Savings: range=[-30022.0090, -10012.1985]
  Rehab_Time_Reduction: range=[-8.6972, -3.2346]


In [30]:
preference_profiles = {
    'Balanced': {
        'Pain_Reduction': 0.25,
        'Short_Term_Pain': 0.15,
        'Functional_Improvement': 0.25,
        'Cost_Savings': 0.20,
        'Rehab_Time_Reduction': 0.15
    },
    'Pain_Focused': {
        'Pain_Reduction': 0.50,
        'Short_Term_Pain': 0.30,
        'Functional_Improvement': 0.10,
        'Cost_Savings': 0.05,
        'Rehab_Time_Reduction': 0.05
    },
    'Cost_Conscious': {
        'Pain_Reduction': 0.15,
        'Short_Term_Pain': 0.10,
        'Functional_Improvement': 0.15,
        'Cost_Savings': 0.45,
        'Rehab_Time_Reduction': 0.15
    },
    'Time_Sensitive': {
        'Pain_Reduction': 0.20,
        'Short_Term_Pain': 0.10,
        'Functional_Improvement': 0.20,
        'Cost_Savings': 0.10,
        'Rehab_Time_Reduction': 0.40
    }
}

for profile_name, weights in preference_profiles.items():
    utility = np.zeros(len(df_work))
    
    for benefit_name, weight in weights.items():
        if benefit_name in benefits_normalized:
            utility += weight * benefits_normalized[benefit_name]
    
    prob_surgery_preferred = (utility > 0.5).mean()
    
    print(f"\n{profile_name}:")
    print(f"  Mean utility: {utility.mean():.3f}")
    print(f"  Median utility: {np.median(utility):.3f}")
    print(f"  P(Surgery preferred): {prob_surgery_preferred:.2%}")
    print(f"  Utility range: [{utility.min():.3f}, {utility.max():.3f}]")


Balanced:
  Mean utility: 0.488
  Median utility: 0.487
  P(Surgery preferred): 44.95%
  Utility range: [0.215, 0.776]

Pain_Focused:
  Mean utility: 0.480
  Median utility: 0.478
  P(Surgery preferred): 46.28%
  Utility range: [0.076, 0.902]

Cost_Conscious:
  Mean utility: 0.493
  Median utility: 0.492
  P(Surgery preferred): 47.38%
  Utility range: [0.223, 0.756]

Time_Sensitive:
  Mean utility: 0.489
  Median utility: 0.488
  P(Surgery preferred): 45.87%
  Utility range: [0.186, 0.797]


## SECTION 11: Final Output Generation

In [31]:
out_rows = {}
if 'PAT_ID' in df_work.columns:
    out_rows['PAT_ID'] = df_work['PAT_ID'].values
else:
    out_rows['PAT_ID'] = np.arange(len(df_work))

for k, outcome in enumerate(outcome_cols, start=1):
    te_name = f"TE{k}"
    mu1_name = f"mu1_{k}"
    mu0_name = f"mu0_{k}"

    if outcome not in all_predictions:
        out_rows[te_name] = np.nan
        out_rows[mu1_name] = np.nan
        out_rows[mu0_name] = np.nan
        continue

    mu1 = all_predictions[outcome]['mu1']
    mu0 = all_predictions[outcome]['mu0']

    if outcome == 'Y4':
        mu1_orig = expm1(mu1)
        mu0_orig = expm1(mu0)
        te_orig = mu1_orig - mu0_orig
    else:
        mu1_orig = mu1
        mu0_orig = mu0
        te_orig = all_predictions[outcome]['te']

    out_rows[mu1_name] = mu1_orig
    out_rows[mu0_name] = mu0_orig
    out_rows[te_name] = te_orig

patient_te_df = pd.DataFrame(out_rows)

export_path = "patient_specific_TE.csv"
patient_te_df.to_csv(export_path, index=False)
print(f"Saved patient-level TE table to: {export_path}")

print("\nFirst 5 rows:")
display(patient_te_df.head())

te_only_cols = ['PAT_ID'] + [f"TE{i}" for i in range(1,6)]
patient_te_only_df = patient_te_df[te_only_cols].copy()


Saved patient-level TE table to: patient_specific_TE.csv

First 5 rows:


,PAT_ID,mu1_1,mu0_1,TE1,mu1_2,mu0_2,TE2,mu1_3,mu0_3,TE3,mu1_4,mu0_4,TE4,mu1_5,mu0_5,TE5
0,1,0.576001,0.402932,0.173069,0.595723,0.802046,-0.206323,0.632654,0.298738,0.333916,21333.678016,1998.324155,19335.353861,10.837676,3.990629,6.847047
1,2,0.766012,0.402913,0.363099,0.458695,0.802872,-0.344177,0.592288,0.297460,0.294828,26657.990549,2000.034246,24657.956303,9.847811,3.991209,5.856602
2,3,0.795338,0.406134,0.389205,0.457668,0.800988,-0.343320,0.595505,0.302630,0.292875,26660.075398,1999.199524,24660.875874,11.137086,4.134564,7.002522
3,4,0.593127,0.398180,0.194947,0.695674,0.802097,-0.106422,0.538746,0.305420,0.233326,18665.878606,1999.217645,16666.660960,10.143620,3.956228,6.187393
4,5,0.669313,0.398826,0.270487,0.661409,0.797870,-0.136461,0.408283,0.300445,0.107838,14664.458923,1997.453934,12667.004989,8.341442,4.038276,4.303166


In [32]:
output_dir = Path('/home/kle-home-lab/code/HA-workspace/SC_Competition/')
output_dir.mkdir(parents=True, exist_ok=True)

csv_path = output_dir / 'patient_specific_TE.csv'
patient_te_df.to_csv(csv_path, index=False)
print(f"\nSaved CDS-ready patient table: {csv_path}")

drivers_path = output_dir / 'clinical_heterogeneity_drivers.csv'
if 'drivers_for_slides' in globals():
    drivers_for_slides.to_csv(drivers_path, index=False)
    print(f"Saved heterogeneity drivers: {drivers_path}")



Saved CDS-ready patient table: /home/kle-home-lab/code/HA-workspace/SC_Competition/patient_specific_TE.csv
Saved heterogeneity drivers: /home/kle-home-lab/code/HA-workspace/SC_Competition/clinical_heterogeneity_drivers.csv


In [33]:
for outcome, best_est in best_model_per_outcome.items():
    r = results[outcome][best_est]['R_loss_mean']
    s = results[outcome][best_est]['training_scale']
    print(f"  {outcome}: {best_est} | R-loss={r:.6f} (scale={s})")

if 'balance_df' in globals():
    pct_balanced = (balance_df['abs_SMD_after'] < 0.1).mean() * 100.0
    print(f"\nBalance after IPW: {pct_balanced:.1f}% of baseline clinical numeric features have |SMD|<0.1")

print(patient_te_df[['PAT_ID'] + [f'TE{i}' for i in range(1,6)]].head())


  Y1: S-learner | R-loss=0.003784 (scale=original)
  Y2: S-learner | R-loss=0.003626 (scale=original)
  Y3: S-learner | R-loss=0.003767 (scale=original)
  Y4: S-learner | R-loss=0.021625 (scale=log1p)
  Y5: S-learner | R-loss=2.238164 (scale=original)

Balance after IPW: 73.0% of baseline clinical numeric features have |SMD|<0.1
   PAT_ID       TE1       TE2       TE3           TE4       TE5
0       1  0.173069 -0.206323  0.333916  19335.353861  6.847047
1       2  0.363099 -0.344177  0.294828  24657.956303  5.856602
2       3  0.389205 -0.343320  0.292875  24660.875874  7.002522
3       4  0.194947 -0.106422  0.233326  16666.660960  6.187393
4       5  0.270487 -0.136461  0.107838  12667.004989  4.303166
